# Nettoyage des données — Delivery ETA (Bronze → Silver)

Objectif : partir du fichier brut `delivery_data.csv` (45 593 lignes, 20 colonnes) et produire un dataset propre, prêt pour l'EDA et la modélisation.

## 1. Imports et chemins

In [1]:
import pandas as pd
import numpy as np
from deliveryeta.paths import DELIVERY_DATA_B, DELIVERY_DATA_S

## 2. Chargement et audit initial

In [2]:
df_raw = pd.read_csv(DELIVERY_DATA_B)
print("Shape :", df_raw.shape)
df_raw.info()
df_raw.head()

Shape : (45593, 20)
<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  str    
 9   Time_Orderd                  45593 non-null  str    
 10  Time_Order_picked            45593 non-null  str    
 11  Weatherconditions            45593 non-null  str    
 12  Road_traffic_density         45593 non-null  str    
 13  Vehicle

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


### Constat
Les colonnes numériques attendues (`Delivery_person_Age`, `Delivery_person_Ratings`, `multiple_deliveries`, `Time_taken(min)`) sont chargées en texte. Un aperçu brut confirme la présence de texte parasite.

In [3]:
print(repr(df_raw.loc[0, "ID"]))
print(repr(df_raw.loc[0, "Weatherconditions"]))
print(repr(df_raw.loc[0, "Time_taken(min)"]))

'0x4607 '
'conditions Sunny'
'(min) 24'


## 3. Nettoyage du texte et normalisation des valeurs manquantes

Avant tout typage, on retire les espaces parasites et on remplace le marqueur `"NaN"` littéral (ex. `"conditions NaN"`) par un vrai `NaN` pandas — sans ça, `.isna()` sous-estime les valeurs manquantes.

In [4]:
df = df_raw.copy()

text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
for column in text_columns:
    df[column] = df[column].astype(str).str.strip()

df["Weatherconditions"] = df["Weatherconditions"].str.replace("conditions ", "", regex=False)
df = df.replace("NaN", np.nan)

print("Valeurs manquantes détectées après normalisation :")
print(df.isna().sum())

Valeurs manquantes détectées après normalisation :
ID                                0
Delivery_person_ID                0
Delivery_person_Age            1854
Delivery_person_Ratings        1908
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Delivery_location_longitude       0
Order_Date                        0
Time_Orderd                    1731
Time_Order_picked                 0
Weatherconditions               616
Road_traffic_density            601
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries             993
Festival                        228
City                           1200
Time_taken(min)                   0
dtype: int64


## 4. Nettoyage de la variable cible

`Time_taken(min)` contient des valeurs comme `"(min) 24"`. Extraction du nombre par regex.

In [5]:
df["Time_taken(min)"] = df["Time_taken(min)"].str.extract(r"(\d+)").astype(float)
print(df["Time_taken(min)"].describe())
print("Valeurs manquantes dans la cible :", df["Time_taken(min)"].isna().sum())

count    45593.000000
mean        26.294607
std          9.383806
min         10.000000
25%         19.000000
50%         26.000000
75%         32.000000
max         54.000000
Name: Time_taken(min), dtype: float64
Valeurs manquantes dans la cible : 0


## 5. Réparation des coordonnées GPS

Deux anomalies détectées :
1. **Erreurs de signe** sur `Restaurant_latitude`/`Restaurant_longitude` (431 et 162 lignes négatives) — impossible pour un dataset 100% en Inde. Corrigées avec `abs()`.
2. **Coordonnées placeholder** proches de zéro sur les 4 colonnes à la fois (3 640 lignes) — non réparables, ces lignes sont supprimées.

In [6]:
gps_columns = ["Restaurant_latitude", "Restaurant_longitude",
               "Delivery_location_latitude", "Delivery_location_longitude"]

print("Latitudes restaurant négatives :", (df["Restaurant_latitude"] < 0).sum())
print("Longitudes restaurant négatives :", (df["Restaurant_longitude"] < 0).sum())

placeholder_mask = (df[gps_columns].abs() < 0.5).all(axis=1)
print("Lignes placeholder (~0,~0,~0,~0) :", placeholder_mask.sum())

df["Restaurant_latitude"] = df["Restaurant_latitude"].abs()
df["Restaurant_longitude"] = df["Restaurant_longitude"].abs()

rows_before = len(df)
df = df.loc[~placeholder_mask].reset_index(drop=True)
print(f"Lignes : {rows_before} -> {len(df)} ({rows_before - len(df)} supprimées)")

Latitudes restaurant négatives : 431
Longitudes restaurant négatives : 162
Lignes placeholder (~0,~0,~0,~0) : 3640
Lignes : 45593 -> 41953 (3640 supprimées)


## 6. Notes de livreur invalides

Une note doit être ≤ 5. Les valeurs supérieures sont des erreurs de saisie : elles sont invalidées (mises à `NaN`) puis imputées avec les autres valeurs manquantes de la colonne.

In [7]:
ratings_numeric = pd.to_numeric(df["Delivery_person_Ratings"], errors="coerce")
print("Notes > 5 :", (ratings_numeric > 5).sum())
df["Delivery_person_Ratings"] = ratings_numeric.where(ratings_numeric <= 5, np.nan)

Notes > 5 : 46


## 7. Typage numérique des colonnes restantes

In [8]:
df["Delivery_person_Age"] = pd.to_numeric(df["Delivery_person_Age"], errors="coerce")
df["multiple_deliveries"] = pd.to_numeric(df["multiple_deliveries"], errors="coerce")
print(df[["Delivery_person_Age", "multiple_deliveries"]].describe())

       Delivery_person_Age  multiple_deliveries
count         40234.000000         41048.000000
mean             29.563330             0.745249
std               5.812361             0.572152
min              15.000000             0.000000
25%              25.000000             0.000000
50%              30.000000             1.000000
75%              35.000000             1.000000
max              50.000000             3.000000


## 8. Imputation des valeurs manquantes

- **Numériques** (`Delivery_person_Age`, `Delivery_person_Ratings`, `multiple_deliveries`) → médiane, robuste aux valeurs extrêmes.
- **Catégorielles** (`Weatherconditions`, `Road_traffic_density`, `Festival`, `City`) → mode.

Toutes les colonnes concernées ont moins de 5 % de valeurs manquantes : l'imputation est préférable à la suppression des lignes.

In [9]:
for col in ["Delivery_person_Age", "Delivery_person_Ratings", "multiple_deliveries"]:
    median_val = df[col].median()
    n_missing = df[col].isna().sum()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: {n_missing} valeurs imputées (médiane = {median_val})")

for col in ["Weatherconditions", "Road_traffic_density", "Festival", "City"]:
    mode_val = df[col].mode()[0]
    n_missing = df[col].isna().sum()
    df[col] = df[col].fillna(mode_val)
    print(f"{col}: {n_missing} valeurs imputées (mode = '{mode_val}')")

print("\nValeurs manquantes restantes :", df.isna().sum().sum())

Delivery_person_Age: 1719 valeurs imputées (médiane = 30.0)
Delivery_person_Ratings: 1809 valeurs imputées (médiane = 4.7)
multiple_deliveries: 905 valeurs imputées (médiane = 1.0)
Weatherconditions: 569 valeurs imputées (mode = 'Fog')
Road_traffic_density: 555 valeurs imputées (mode = 'Low')
Festival: 215 valeurs imputées (mode = 'No')
City: 1114 valeurs imputées (mode = 'Metropolitian')

Valeurs manquantes restantes : 1600


## 9. Colonnes temporelles

`Order_Date` est convertie en datetime. Pour `Time_Orderd` (heure de commande, ~3,8 % manquante), on reconstruit la valeur manquante à partir de `Time_Order_picked` moins le temps de préparation médian observé sur les lignes complètes.

In [10]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%d-%m-%Y")

order_dt = pd.to_datetime(df["Time_Orderd"], format="%H:%M:%S", errors="coerce")
picked_dt = pd.to_datetime(df["Time_Order_picked"], format="%H:%M:%S", errors="coerce")
prep_minutes = ((picked_dt - order_dt).dt.total_seconds() / 60).mod(1440)
median_prep = prep_minutes.median()
print("Temps de préparation médian :", median_prep, "min")

missing_mask = df["Time_Orderd"].isna()
estimated = picked_dt[missing_mask] - pd.to_timedelta(median_prep, unit="m")
df.loc[missing_mask, "Time_Orderd"] = estimated.dt.strftime("%H:%M:%S")

print(f"{missing_mask.sum()} heures de commande reconstruites")
print("Valeurs manquantes restantes dans Time_Orderd :", df["Time_Orderd"].isna().sum())

Temps de préparation médian : 10.0 min
1600 heures de commande reconstruites
Valeurs manquantes restantes dans Time_Orderd : 0


## 10. Doublons

In [11]:
dup_rows = df.duplicated().sum()
dup_no_id = df.drop(columns=["ID"]).duplicated().sum()
print("Doublons (avec ID) :", dup_rows)
print("Doublons (sans ID) :", dup_no_id)
df = df.drop_duplicates()

Doublons (avec ID) : 0
Doublons (sans ID) : 0


## 11. Sélection finale des colonnes

Seules `ID` et `Delivery_person_ID` sont supprimées : ce sont des identifiants sans valeur prédictive. Toutes les autres colonnes sont conservées — aucune anomalie ne justifie de les écarter, et leur pouvoir prédictif réel sera mesuré en EDA plutôt que supposé à l'avance.

In [12]:
df = df.drop(columns=["ID", "Delivery_person_ID"])
print("Shape finale :", df.shape)
df.info()

Shape finale : (41953, 18)
<class 'pandas.DataFrame'>
RangeIndex: 41953 entries, 0 to 41952
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Delivery_person_Age          41953 non-null  float64       
 1   Delivery_person_Ratings      41953 non-null  float64       
 2   Restaurant_latitude          41953 non-null  float64       
 3   Restaurant_longitude         41953 non-null  float64       
 4   Delivery_location_latitude   41953 non-null  float64       
 5   Delivery_location_longitude  41953 non-null  float64       
 6   Order_Date                   41953 non-null  datetime64[us]
 7   Time_Orderd                  41953 non-null  str           
 8   Time_Order_picked            41953 non-null  str           
 9   Weatherconditions            41953 non-null  str           
 10  Road_traffic_density         41953 non-null  str           
 11  Vehicle_condition        

## 12. Export

In [13]:
df.to_csv(DELIVERY_DATA_S, index=False)
print("Sauvegardé :", DELIVERY_DATA_S)

Sauvegardé : C:\Users\Elkerymy-Mohamed\Study-DevAI\DeliveryETA\data\silver\delivery_data.csv


## Résumé

| Étape | Résultat |
|---|---|
| Cible nettoyée | `Time_taken(min)`, 10 à 54 min, 0 manquant |
| Coordonnées GPS | 593 erreurs de signe corrigées, 3 640 lignes placeholder supprimées |
| Notes invalides | 46 lignes (>5) invalidées puis imputées |
| Valeurs manquantes | imputées (médiane / mode), 0 restante |
| Heures de commande manquantes | reconstruites via le temps de préparation médian |
| Doublons | 0 |
| Colonnes supprimées | `ID`, `Delivery_person_ID` |
| **Shape finale** | **41 953 lignes × 18 colonnes** |

Dataset prêt pour l'EDA et le feature engineering.